# LLM Evaluation & Benchmarks — from scratch

How do you measure a thing that can talk its way out of the test? This notebook builds the five
core evaluation primitives **from scratch** on tiny hand-built inputs — no model download, no GPU —
and *proves* each one's key claim with an `assert` before printing it, never on faith:

1. **Perplexity** = `exp(cross-entropy)` — the identity, and a better model has lower PPL.
2. **`pass@k`** — the unbiased estimator `1 − C(n−c,k)/C(n,k)`, and why the naive one is biased **low**.
3. **ECE** — expected calibration error; a calibrated model ≈ 0, an over-confident one is not.
4. **Elo / Bradley–Terry** — recover the true skill ordering from win/loss alone.
5. **LLM-judge position bias** — swap the answer order and watch verdicts flip.

Every function and constant is **imported** from the sibling script `llm_evaluation.py`, so the
numbers here are bit-for-bit the same ones on the concept page and in the figures — there is exactly
one source of truth.

## 0. Setup and device honesty

Most of this notebook is plain numpy / Python `math` — the math is device-independent. Only the
perplexity cross-entropy touches torch. We pin the reproducible trace to **CPU** and print the device
honestly: the printed device must match where the numbers were actually produced.

In [1]:
import math
import numpy as np
import torch

# Canonical source of truth: every function and constant comes from the sibling script.
from llm_evaluation import (
    PPL_VOCAB, HELD_OUT_IDS, GOOD_MODEL_PROBS, BAD_MODEL_PROBS,
    PASS_N, PASS_C, PASS_K_VALUES, PASS_NAIVE_TRIALS, PASS_NAIVE_SEED,
    ECE_N_BINS, ECE_N_SAMPLES, ECE_OVERCONF_GAP, ECE_SEED,
    TRUE_SKILL, ELO_MATCHES, ELO_SEED, ELO_BASE, ELO_K, ELO_SCALE,
    JUDGE_N_PAIRS, JUDGE_BIAS, JUDGE_SEED,
    cross_entropy_nats, perplexity,
    pass_at_k, pass_at_k_naive_mc,
    expected_calibration_error, make_calibration_data,
    expected_score, elo_update, simulate_elo, ranking,
    measure_position_bias,
)

trace_device = 'cpu'  # pin the trace; print honestly
detected = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {trace_device} (detected {detected}; pinned to CPU for reproducibility)')
print('torch:', torch.__version__, '| numpy:', np.__version__)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0 | numpy: 2.4.6


## 1. Perplexity = exp(cross-entropy)

Two toy 'models' assign a next-token distribution to the **same** held-out sequence
`the cat sat on mat <eos>`. The better model puts more probability on the tokens that actually
occur → lower cross-entropy → lower perplexity. We prove `PPL == exp(CE)` by identity, and that the
confident-correct model wins.

In [2]:
good = torch.tensor(GOOD_MODEL_PROBS, device=trace_device)
bad  = torch.tensor(BAD_MODEL_PROBS,  device=trace_device)

ce_good, ce_bad = cross_entropy_nats(good, HELD_OUT_IDS), cross_entropy_nats(bad, HELD_OUT_IDS)
ppl_good, ppl_bad = perplexity(good, HELD_OUT_IDS), perplexity(bad, HELD_OUT_IDS)

# Assert the qualitative claims BEFORE printing the numbers.
assert abs(ppl_good - math.exp(ce_good)) < 1e-9, 'PPL must equal exp(CE) by definition'
assert ppl_good < ppl_bad, 'the confident-correct model must have the lower perplexity'

seq = ' '.join(PPL_VOCAB[i] for i in HELD_OUT_IDS)
print(f"held-out sequence: '{seq}'")
print(f'  GOOD model: CE={ce_good:.4f} nats   PPL={ppl_good:.3f}')
print(f'  BAD  model: CE={ce_bad:.4f} nats   PPL={ppl_bad:.3f}')
print(f'  => PPL == exp(CE) exactly; better model lower PPL ({ppl_good:.3f} < {ppl_bad:.3f}).')

held-out sequence: 'the cat sat on mat <eos>'
  GOOD model: CE=0.3814 nats   PPL=1.464
  BAD  model: CE=1.3863 nats   PPL=4.000
  => PPL == exp(CE) exactly; better model lower PPL (1.464 < 4.000).


The near-uniform model lands at **PPL 4.0** — when it spreads its mass it behaves like guessing among
~4 options, and perplexity reports exactly that branching factor. Perplexity is *interpretable*: it's
the effective number of choices the model hesitates between per token.

## 2. `pass@k` — the unbiased estimator, and the naive bias

For code, a completion either passes the unit tests or not. With `n` samples and `c` correct, the
unbiased `pass@k = 1 − C(n−c,k)/C(n,k)` is the exact chance a *without-replacement* draw of `k` hits a
correct one. The naive *with-replacement* recipe is biased **low**. We sweep `k` and check both the
monotonicity and the boundary identities (`pass@1 = c/n`, `pass@n = 1`).

In [3]:
curve = [pass_at_k(PASS_N, PASS_C, k) for k in PASS_K_VALUES]

# Assert the qualitative claims first.
assert all(b >= a - 1e-12 for a, b in zip(curve, curve[1:])), 'pass@k must be non-decreasing in k'
assert abs(curve[0] - PASS_C / PASS_N) < 1e-9, 'pass@1 must equal c/n'
assert abs(curve[-1] - 1.0) < 1e-9, 'pass@n must be 1.0'

print(f'n={PASS_N} samples, c={PASS_C} correct -> chance >=1 of k passes:')
for k, val in zip(PASS_K_VALUES, curve):
    print(f'  pass@{k:>2} = {val:.3f}  ' + '#' * int(round(val * 40)))

n=20 samples, c=5 correct -> chance >=1 of k passes:
  pass@ 1 = 0.250  ##########
  pass@ 2 = 0.447  ##################
  pass@ 3 = 0.601  ########################
  pass@ 5 = 0.806  ################################
  pass@ 8 = 0.949  ######################################
  pass@10 = 0.984  #######################################
  pass@15 = 1.000  ########################################
  pass@20 = 1.000  ########################################


In [4]:
# The naive with-replacement estimator is biased LOW -- measure the gap, don't assert it.
k_demo = 5
unbiased_k = pass_at_k(PASS_N, PASS_C, k_demo)
naive_k = pass_at_k_naive_mc(PASS_N, PASS_C, k_demo, PASS_NAIVE_TRIALS, PASS_NAIVE_SEED)
assert naive_k < unbiased_k - 1e-3, 'naive estimator must be biased low'
print(f'unbiased pass@{k_demo} = {unbiased_k:.3f}   >   naive (with-replacement) = {naive_k:.3f}')
print(f'the bias (measured): {unbiased_k - naive_k:.3f} -- repeated wrong draws inflate failure.')

unbiased pass@5 = 0.806   >   naive (with-replacement) = 0.761
the bias (measured): 0.045 -- repeated wrong draws inflate failure.


## 3. ECE — is the model's confidence honest?

A model is calibrated if, among the times it says '80% sure', it's right 80% of the time. We synthesize
a **calibrated** stream (accuracy == confidence) and an **over-confident** one (accuracy lags confidence
by 0.15), bin by confidence, and compute ECE = the bin-weighted |accuracy − confidence|
gap. Calibrated ≈ 0; over-confident clearly positive.

In [5]:
conf_cal,  corr_cal  = make_calibration_data(ECE_N_SAMPLES, 0.0,             ECE_SEED)
conf_over, corr_over = make_calibration_data(ECE_N_SAMPLES, ECE_OVERCONF_GAP, ECE_SEED)
ece_cal,  _ = expected_calibration_error(conf_cal,  corr_cal,  ECE_N_BINS)
ece_over, _ = expected_calibration_error(conf_over, corr_over, ECE_N_BINS)

# Assert the qualitative claims first.
assert ece_cal < 0.03,  'calibrated data must have near-zero ECE'
assert ece_over > 0.10, 'over-confident data must have a clearly positive ECE'
assert ece_over > ece_cal, 'over-confident ECE must exceed calibrated ECE'

print(f'calibrated    (gap 0.00): ECE = {ece_cal:.4f}   acc {corr_cal.mean():.3f} vs conf {conf_cal.mean():.3f}')
print(f'over-confident(gap {ECE_OVERCONF_GAP:.2f}): ECE = {ece_over:.4f}   acc {corr_over.mean():.3f} vs conf {conf_over.mean():.3f}')
print(f'  => over-confidence lifts ECE {ece_cal:.4f} -> {ece_over:.4f}.')

calibrated    (gap 0.00): ECE = 0.0115   acc 0.749 vs conf 0.749
over-confident(gap 0.15): ECE = 0.1542   acc 0.595 vs conf 0.749
  => over-confidence lifts ECE 0.0115 -> 0.1542.


ECE turns a *reliability diagram* into one number. Beware: it depends on the bin count and can hide a
model that's over-confident in one region and under-confident in another — always show the diagram too.

## 4. Elo / Bradley–Terry — recover skill from win/loss alone

Chatbot Arena turns pairwise human votes into one rating per model. The expected score is a logistic in
the rating gap — `E_A = 1/(1 + 10^((R_B−R_A)/400))` — and each game nudges ratings by `K·(S−E)`.
First, read off the +400 ⇒ 10× odds convention; then watch Elo recover a known ordering from outcomes.

In [6]:
# The +400 => ~10x odds convention, straight from the logistic.
for gap in (0, 200, 400):
    e = expected_score(gap, 0.0, ELO_SCALE)
    print(f'rating gap +{gap:>3}: expected score E_A = {e:.3f}')
assert abs(expected_score(0, 0) - 0.5) < 1e-9, 'equal ratings -> coin flip'
assert abs(expected_score(ELO_SCALE, 0) - 10/11) < 1e-3, '+400 -> ~0.909 (10:1 odds)'

rating gap +  0: expected score E_A = 0.500
rating gap +200: expected score E_A = 0.760
rating gap +400: expected score E_A = 0.909


In [7]:
# Four agents of KNOWN hidden skill; Elo sees ONLY win/loss and must recover the ordering.
final_ratings, history = simulate_elo(TRUE_SKILL, ELO_MATCHES, ELO_SEED)
recovered = ranking(final_ratings)
true_order = ranking(TRUE_SKILL)

# THE key result: recovered ranking == true ranking, from outcomes alone.
assert recovered == true_order, 'Elo must recover the true skill ordering'

print(f'{ELO_MATCHES} games, flat {ELO_BASE:.0f} start, K={ELO_K:.0f}:')
for name in recovered:
    print(f'  {name}: Elo {final_ratings[name]:7.1f}   (true skill {TRUE_SKILL[name]:.0f})')
print(f'  recovered: {" > ".join(recovered)}   |   true: {" > ".join(true_order)}')

4000 games, flat 1000 start, K=32:
  A: Elo  1355.5   (true skill 1600)
  B: Elo  1080.6   (true skill 1400)
  C: Elo   912.1   (true skill 1200)
  D: Elo   651.9   (true skill 1000)
  recovered: A > B > C > D   |   true: A > B > C > D


The absolute ratings compress toward the center (a tiny 4-agent pool plays too few games to reach the
full 600-point spread), but the **ordering** — all a leaderboard reports — is exactly right. Real Arena
fits Bradley–Terry by maximum likelihood over all votes at once (with confidence intervals); online Elo
is the right mental model and converges to the same order.

## 5. LLM-as-judge — position bias, measured

A strong LLM can score open-ended answers cheaply, but it's biased. The clearest is **position bias**:
it favours whichever answer it sees first. Rate every pair in **both orders** and count flips. A fair
judge never flips (50% first-position win-rate); a biased one flips near-ties and exceeds 50%.

> **Try it — predict before you run.** The biased judge below has `bias=0.15` over 500 pairs. Before running the next cell, **predict**: will the flip-rate (verdicts that change when you swap answer order) be closer to **5%**, **25%**, or **50%**? And will the first-position win-rate land near 0.5, 0.6, or 0.75? Then run it and check.
>
> *Hint:* only **near-ties** flip — when the two answers' true qualities are close enough that a 0.15 nudge tips the verdict. Pairs with a clear winner don't flip. So the flip-rate tracks "what fraction of pairs are near-ties," not the bias size directly.

In [8]:
unbiased = measure_position_bias(JUDGE_N_PAIRS, bias=0.0,        seed=JUDGE_SEED)
biased   = measure_position_bias(JUDGE_N_PAIRS, bias=JUDGE_BIAS, seed=JUDGE_SEED)

# Assert the qualitative claims first.
assert unbiased.flip_rate == 0.0, 'a fair judge never flips on order swap'
assert abs(unbiased.first_pos_winrate - 0.5) < 1e-9, 'fair judge: 50% first-position wins'
assert biased.flip_rate > 0.0, 'position bias must produce order-dependent flips'
assert biased.first_pos_winrate > 0.5, 'position bias must favour the first-shown answer'

print(f'{JUDGE_N_PAIRS} answer pairs, rated in both orders:')
print(f'  fair   judge (bias=0.0):   flip-rate {unbiased.flip_rate:.3f}   first-pos win-rate {unbiased.first_pos_winrate:.3f}')
print(f'  biased judge (bias={JUDGE_BIAS}):  flip-rate {biased.flip_rate:.3f}   first-pos win-rate {biased.first_pos_winrate:.3f}')
print(f'  => position bias flips {biased.flip_rate:.1%} of verdicts; first-position win-rate {biased.first_pos_winrate:.1%}.')

500 answer pairs, rated in both orders:
  fair   judge (bias=0.0):   flip-rate 0.000   first-pos win-rate 0.500
  biased judge (bias=0.15):  flip-rate 0.270   first-pos win-rate 0.635
  => position bias flips 27.0% of verdicts; first-position win-rate 63.5%.


## Recap

Five primitives, each proven not asserted: **perplexity = exp(CE)**; **pass@k** unbiased vs the
biased-low naive form; **ECE** separating a calibrated from an over-confident model; **Elo** recovering
a true ranking from win/loss alone; and **LLM-judge position bias** as a measurable flip-rate. The page
ties them into one **evaluation portfolio** — intrinsic, capability, preference, and trust — because no
single number survives Goodhart and contamination on its own.